# Impala67 – Bild-Modell: Mathe-Training in Google Colab

Startet vom fertigen Bild-Modell **de-htr-web-v2** (Apache-2.0) und bringt ihm Formeln bei, ohne Text zu verlernen:
jeder Batch ist zu 35 % Mathe (MathWriting + synthetische Rechnungen) und zu 65 % Text (ScaDS + fhswf als Scan, dazu gezeichnete Sätze mit Leerzeichen).

1. **Laufzeit → Laufzeittyp ändern → L4 GPU** (sonst T4).
2. **Laufzeit → Alle ausführen**.
3. Beim ersten Mal: Google Drive freigeben und `benchmark_test_strokes.json` hochladen (liegt lokal in `handwriting/data/`).
   Optional `my_handwriting_samples.json` nach `Impala67-Handschrift/` legen: deine Zeilen werden dann **nur gemessen**, nie trainiert.

Datensätze (einmalig, danach als Cache in Drive): ScaDS.AI German Lines (Zenodo, CC BY 4.0, 244 MB), fhswf/german_handwriting (Hugging Face, AFL-3.0, 1,8 GB), Google MathWriting (CC BY 4.0, 3 GB).

Zwischenstände liegen in Drive unter `Impala67-Handschrift/checkpoints-bild/<LAUF>/`. Bricht die Sitzung ab: erneut **Alle ausführen**, es geht beim letzten Stand weiter. Neuer Lauf von vorn: `LAUF` in der Trainingszelle ändern.

Ergebnis-Zeilen im Log: `sel_text`/`test_text` = Text fremder Schreiber (ScaDS, nie trainiert), `sel_math`/`test_math` = Formeln fremder Schreiber (MathWriting test), `sel_ink`/`test_ink` = gezeichneter Text fremder Schreiber (BRUSH-Cache aus Drive), `eigene` = deine Handschrift. `best.pt` wird nur nach `sel_*` gewählt; Text darf dabei höchstens 1 Punkt schlechter werden als beim Start.

In [ ]:
import os, subprocess, shutil, hashlib, glob
import torch
assert torch.cuda.is_available(), 'Keine GPU! Laufzeit -> Laufzeittyp ändern -> L4 oder T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Impala67-Handschrift'
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
REPO = '/content/Impala67'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q --depth 1 https://github.com/prinzpuma/Impala67.git {REPO}
!pip install -q onnx onnxruntime onnxscript scikit-image pillow pyarrow huggingface_hub
HW = f'{REPO}/handwriting'
os.makedirs(f'{HW}/data/mw_download', exist_ok=True)
if os.path.exists(f'{DRIVE}/code.zip'):
    !unzip -o -q {DRIVE}/code.zip -d {HW}
    print('Aktualisierter Code aus Drive/code.zip übernommen')

In [ ]:
# Startmodell de-htr-web-v2 (fp32, 40 MB) und eigene Messzeilen
from huggingface_hub import hf_hub_download
DEHTR = f'{HW}/data/dehtr'
for name in ('model-fp32.onnx', 'config.json'):
    hf_hub_download('naeyn/de-htr-web-v2', name, local_dir=DEHTR)

bench_drive = f'{DRIVE}/benchmark_test_strokes.json'
if not os.path.exists(bench_drive):
    from google.colab import files
    print('Bitte handwriting/data/benchmark_test_strokes.json hochladen')
    up = files.upload()
    with open(bench_drive, 'wb') as f:
        f.write(next(iter(up.values())))
shutil.copy(bench_drive, f'{HW}/data/benchmark_test_strokes.json')
SAMPLES = f'{HW}/data/my_handwriting_samples.json'
if os.path.exists(f'{DRIVE}/my_handwriting_samples.json'):
    shutil.copy(f'{DRIVE}/my_handwriting_samples.json', SAMPLES)
print('Startmodell und Messzeilen bereit')

In [ ]:
# Google MathWriting (CC BY 4.0). Liegt der fertige Cache in Drive, entfällt der 3-GB-Download.
caches = glob.glob(f'{DRIVE}/mathwriting-2024-*-v4.cache.pkl')  # muss zu CACHE_VERSION in mathwriting_loader.py passen
for c in caches:
    shutil.copy(c, f'{HW}/data/mw_download/')

# Der test-Teil (fremde Schreiber, nur zum Messen) hat einen eigenen Cache; fehlt er, einmalig neu laden
has_test = any('-test-' in c for c in caches)
if not (caches and has_test) and not os.path.isdir(f'{HW}/data/mw_download/mathwriting-2024'):
    tgz = '/content/mathwriting-2024.tgz'
    !wget -q -c -O {tgz} https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz
    md5 = hashlib.md5()
    with open(tgz, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            md5.update(chunk)
    assert md5.hexdigest() == 'f2d59c44a545347a5f67ac70fef7a13d', 'Download beschädigt – Zelle erneut ausführen'
    !tar -xzf {tgz} -C {HW}/data/mw_download
    os.remove(tgz)

# Caches bauen (einmalig einige Minuten) und in Drive sichern
!cd {HW} && python -c "from mathwriting_loader import MathWritingDataset as M; M(); M(split='test')"
for c in glob.glob(f'{HW}/data/mw_download/*.cache.pkl'):
    if not os.path.exists(f'{DRIVE}/{os.path.basename(c)}'):
        shutil.copy(c, DRIVE)
print('MathWriting bereit')

In [ ]:
# ScaDS.AI German Lines (CC BY 4.0) und fhswf/german_handwriting (AFL-3.0) als Bild-Cache in Drive
for name in ('scads-lines-v1.pkl', 'fhswf-lines-v1.pkl'):
    if os.path.exists(f'{DRIVE}/{name}'):
        shutil.copy(f'{DRIVE}/{name}', f'{HW}/data/')

if not os.path.exists(f'{HW}/data/scads-lines-v1.pkl'):
    tgz = '/content/scads.tar.gz'
    !wget -q -c -O {tgz} "https://zenodo.org/records/18301532/files/scadsai_german_handwriting_line_word_level_v01.tar.gz?download=1"
    os.makedirs('/content/scads', exist_ok=True)
    !tar -xzf {tgz} -C /content/scads
    !cd {HW} && python -c "from image_datasets import load_scads; load_scads('/content/scads', 'data')"
    shutil.copy(f'{HW}/data/scads-lines-v1.pkl', DRIVE)

if not os.path.exists(f'{HW}/data/fhswf-lines-v1.pkl'):
    from huggingface_hub import snapshot_download
    snapshot_download('fhswf/german_handwriting', repo_type='dataset', allow_patterns='data/*.parquet', local_dir='/content/fhswf')
    !cd {HW} && python -c "from image_datasets import load_fhswf; load_fhswf('/content/fhswf', 'data')"
    shutil.copy(f'{HW}/data/fhswf-lines-v1.pkl', DRIVE)
print('Text-Daten bereit')

In [ ]:
# Training. Zwischenstände gehen nach Drive (last.pt bei jeder Messung, best.pt bei Verbesserung).
LAUF = 'lauf-3'  # neuer Name = neuer Lauf von vorn; gleicher Name = weitermachen
ckpt_dir = f'{DRIVE}/checkpoints-bild/{LAUF}'
os.makedirs(ckpt_dir, exist_ok=True)
# BRUSH-Testschreiber (gezeichneter Text, nur zum Messen): brush_loader sucht den Cache neben dem Checkpoint-Ordner
if os.path.exists(f'{DRIVE}/brush-val-v1.cache.pkl'):
    shutil.copy(f'{DRIVE}/brush-val-v1.cache.pkl', os.path.dirname(ckpt_dir))
args = ['python', 'train_image.py', '--dehtr', DEHTR, '--samples', SAMPLES]
env = dict(os.environ, HW_CHECKPOINT_DIR=ckpt_dir, PYTHONUNBUFFERED='1')
proc = subprocess.Popen(args, cwd=HW, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(f'{DRIVE}/train-bild.log', 'a') as log:
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
assert proc.wait() == 0, 'Training mit Fehler beendet – siehe Ausgabe oben'

In [ ]:
# Fertiges Bild-Modell (int8, ca. 10 MB) nach Drive kopieren und herunterladen
shutil.copy(f'{HW}/model.onnx', f'{DRIVE}/model-bild.onnx')
print('Modell in Drive gespeichert:', f'{DRIVE}/model-bild.onnx')
from google.colab import files
files.download(f'{HW}/model.onnx')